In [1]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal
from sklearn.metrics import pairwise_distances
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

from matplotlib import cm, colormaps
from PIL import Image
import seaborn as sns

sns.set_style("whitegrid")

# Reproducibility
random.seed(42)
np.random.seed(42)

# ============================================================================
# PATH CONFIGURATION (Simplified & Clean)
# ============================================================================
workspace_root = Path.cwd()
input_csv = workspace_root / "gpt-120-enriched.csv"
results_dir = workspace_root / "results"

if not input_csv.exists():
    raise FileNotFoundError(f"Input CSV not found: {input_csv}")

results_dir.mkdir(parents=True, exist_ok=True)

print(f"Input CSV: {input_csv}")
print(f"Results directory: {results_dir}\n")

# ============================================================================
# DATA LOADING & PREPROCESSING
# ============================================================================
def load_input_csv(csv_path: Path) -> pd.DataFrame:
    """Load CSV and normalize column names to lowercase."""
    df_in = pd.read_csv(csv_path)
    df_in.columns = [str(col).strip().lower() for col in df_in.columns]
    return df_in


def clean_abstract(text: str) -> str:
    """Remove URLs and normalize whitespace in abstracts."""
    text = re.sub(r"http\S+|www\S+", "", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Define stopwords and common phrases to remove
custom_stopwords = {
    "evidencenet", "fakenewsnet", "factify", "politifact", "snopes",
    "rumoureval", "cnn", "weibo", "facebook", "wikipedia"
}
stopword_pattern = r'\b(' + '|'.join(re.escape(word) for word in custom_stopwords) + r')\b'
common_phrases = ["this paper", "in this paper", "in this study", "this work"]

# Load and clean data
df = load_input_csv(input_csv)
df = df[df["abstract"].notna()].copy()
df = df[df["abstract"].astype(str).str.strip() != ""].copy()
df = df[df["abstract"].astype(str).str.split().str.len() > 20].copy()

# Clean abstracts
cleaned_abstracts = df["abstract"].apply(clean_abstract)
cleaned_abstracts = cleaned_abstracts.str.replace(stopword_pattern, '', case=False, regex=True)
for phrase in common_phrases:
    cleaned_abstracts = cleaned_abstracts.str.replace(phrase, '', case=False, regex=False)

df["abstract_clean"] = cleaned_abstracts.str.strip()
df = df[df["abstract_clean"].str.strip() != ""].copy()
df = df.drop_duplicates(subset="abstract_clean").reset_index(drop=True)
df["document_id"] = df.index

text_series = df["abstract_clean"].copy()

# Embedding models to process
embedding_model_names = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "distiluse-base-multilingual-cased-v1",
    "paraphrase-MiniLM-L6-v2"
]

print(f"Documents after preprocessing: {len(df)}")
print(f"Embedding models to process: {embedding_model_names}\n")

/Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/gpt-oss-120b/topic-modeling/bertopic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input CSV: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/gpt-oss-120b/topic-modeling/bertopic/gpt-120-enriched.csv
Results directory: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/gpt-oss-120b/topic-modeling/bertopic/results

Documents after preprocessing: 812
Embedding models to process: ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'distiluse-base-multilingual-cased-v1', 'paraphrase-MiniLM-L6-v2']



In [2]:
# ============================================================================
# BERTOPIC MODEL TRAINING & FITTING
# ============================================================================

# Extract metadata fields if available
metadata_fields = [col for col in ["id", "doi", "title", "year"] if col in df.columns]
df_metadata = df[metadata_fields].copy() if metadata_fields else pd.DataFrame(index=df.index)
df_metadata["doc_index"] = df_metadata.index

# Process each embedding model
for embedding_model_name in embedding_model_names:
    print(f"\n{'='*60}")
    print(f"Processing: {embedding_model_name}")
    print('='*60)

    # Create model-specific directory
    normalized_model_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / normalized_model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # Define paths for model artifacts
    model_file = model_dir / "bertopic_model.pkl"
    csv_file = model_dir / "1_documents_dataset_mapping.csv"
    embeddings_file = model_dir / "documents_embeddings.npy"

    # Check if model already exists
    if model_file.exists() and csv_file.exists() and embeddings_file.exists():
        print("✓ Loading existing model and embeddings...")
        topic_model = BERTopic.load(str(model_file))
        df_out = pd.read_csv(csv_file)
        embeddings = np.load(embeddings_file)

    else:
        print("→ Training new BERTopic model...")

        # Step 1: Generate embeddings
        sentence_model = SentenceTransformer(embedding_model_name)
        embeddings = sentence_model.encode(text_series.tolist(), show_progress_bar=True)
        np.save(embeddings_file, embeddings)

        # Step 2: Configure clustering & reduction models
        umap_model = UMAP(n_neighbors=15, n_components=50, metric='cosine', random_state=42)
        hdbscan_model = HDBSCAN(
            min_cluster_size=5,
            min_samples=2,
            metric='euclidean',
            cluster_selection_method='leaf'
        )
        vectorizer_model = CountVectorizer(stop_words="english")

        # Step 3: Train BERTopic model
        topic_model = BERTopic(
            embedding_model=None,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            verbose=True
        )

        topics, probs = topic_model.fit_transform(text_series.tolist(), embeddings)

        # Step 4: Save results
        df_out = pd.DataFrame({
            "document_id": list(range(len(text_series))),
            "text": text_series,
            "topic_id": topics,
            "probability": [float(p) if p is not None else None for p in probs]
        })

        # Add metadata
        df_out = df_out.merge(df_metadata, left_on="document_id", right_on="doc_index", how="left")

        # Persist model and data
        topic_model.save(str(model_file))
        df_out.to_csv(csv_file, index=False)

    print(f"✓ Model ready for {embedding_model_name}")


Processing: all-MiniLM-L6-v2
→ Training new BERTopic model...


Batches: 100%|██████████| 26/26 [00:07<00:00,  3.50it/s]
2026-05-03 20:45:30,985 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 20:45:39,536 - BERTopic - Dimensionality - Completed ✓
2026-05-03 20:45:39,537 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 20:45:39,563 - BERTopic - Cluster - Completed ✓
2026-05-03 20:45:39,568 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 20:45:39,680 - BERTopic - Representation - Completed ✓
2026-05-03 20:45:39,868 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ Model ready for all-MiniLM-L6-v2

Processing: all-mpnet-base-v2
→ Training new BERTopic model...


Batches: 100%|██████████| 26/26 [00:43<00:00,  1.69s/it]
2026-05-03 20:46:31,156 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 20:46:33,544 - BERTopic - Dimensionality - Completed ✓
2026-05-03 20:46:33,545 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 20:46:33,568 - BERTopic - Cluster - Completed ✓
2026-05-03 20:46:33,570 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 20:46:33,681 - BERTopic - Representation - Completed ✓
2026-05-03 20:46:33,860 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ Model ready for all-mpnet-base-v2

Processing: distiluse-base-multilingual-cased-v1
→ Training new BERTopic model...


Batches: 100%|██████████| 26/26 [00:07<00:00,  3.34it/s]
2026-05-03 20:47:10,080 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 20:47:12,404 - BERTopic - Dimensionality - Completed ✓
2026-05-03 20:47:12,406 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 20:47:12,433 - BERTopic - Cluster - Completed ✓
2026-05-03 20:47:12,436 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 20:47:12,549 - BERTopic - Representation - Completed ✓
2026-05-03 20:47:12,734 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ Model ready for distiluse-base-multilingual-cased-v1

Processing: paraphrase-MiniLM-L6-v2
→ Training new BERTopic model...


Batches: 100%|██████████| 26/26 [00:03<00:00,  8.32it/s]
2026-05-03 20:47:22,410 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 20:47:24,701 - BERTopic - Dimensionality - Completed ✓
2026-05-03 20:47:24,703 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 20:47:24,728 - BERTopic - Cluster - Completed ✓
2026-05-03 20:47:24,730 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 20:47:24,838 - BERTopic - Representation - Completed ✓
2026-05-03 20:47:25,009 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ Model ready for paraphrase-MiniLM-L6-v2


In [3]:
# ============================================================================
# VERIFY MODELS & DISPLAY SUMMARY
# ============================================================================

for embedding_model_name in embedding_model_names:
    normalized_model_name = embedding_model_name.replace("/", "_")
    model_file = results_dir / normalized_model_name / "bertopic_model.pkl"

    if model_file.exists():
        try:
            model = BERTopic.load(str(model_file))
            topic_info = model.get_topic_info()
            num_topics = len(topic_info)
            topic_sizes = topic_info["Count"].tolist()

            print(f"\n{embedding_model_name}")
            print(f"  → Topics: {num_topics}")
            print(f"  → Total documents: {sum(topic_sizes)}")

        except Exception as e:
            print(f"✗ Error loading {embedding_model_name}: {e}")
    else:
        print(f"✗ Model not found: {model_file}")


all-MiniLM-L6-v2
  → Topics: 58
  → Total documents: 812

all-mpnet-base-v2
  → Topics: 57
  → Total documents: 812

distiluse-base-multilingual-cased-v1
  → Topics: 58
  → Total documents: 812

paraphrase-MiniLM-L6-v2
  → Topics: 52
  → Total documents: 812


In [4]:
# ============================================================================
# EXPORT TOPICS & GENERATE SUMMARIES
# ============================================================================

def suggest_topic_name(top_words: list[str]) -> str:
    """
    Suggests a short, human-readable name based on top words.
    Uses domain-specific patterns first, then falls back to keyword combinations.
    """
    pattern_dict = [
        ({"fake", "fakenews", "propaganda", "disinformation", "misinformation"},
         "Misinformation & Propaganda"),
        ({"debunking", "factcheck", "veracity", "refuting", "factchecking"},
         "Fact-Checking & Verification"),
        ({"credibility", "trust", "reliable", "confidence"},
         "Credibility Assessment"),
        ({"journalism", "journalistic", "media", "reporting"},
         "Journalism & Media Studies"),
        ({"rumors", "rumor", "rumour", "retweets", "twitter", "microblogs"},
         "Rumor on Social Media"),
        ({"crowdsourcing", "crowd", "annotation", "human"},
         "Crowdsourced Analysis"),
        ({"nlp", "classifiers", "embedding", "bayes", "transformer"},
         "NLP Methods"),
    ]

    lower_words = {w.lower() for w in top_words}

    for word_set, label in pattern_dict:
        if lower_words & word_set:
            return label

    # Fallback: combine significant keywords
    generic_terms = {"news", "paper", "model", "dataset", "data", "using",
                     "analysis", "study", "approach", "method", "methods"}
    significant = [w for w in top_words if w.lower() not in generic_terms]
    if len(significant) >= 2:
        return ", ".join(significant[:2])
    elif significant:
        return significant[0]
    else:
        return ", ".join(top_words[:2])


for embedding_model_name in embedding_model_names:
    print(f"\n{'='*60}")
    print(f"Exporting topics: {embedding_model_name}")
    print('='*60)

    normalized_model_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / normalized_model_name
    model_file = model_dir / "bertopic_model.pkl"
    csv_file = model_dir / "1_documents_dataset_mapping.csv"
    topics_dir = model_dir / "topics"
    topics_dir.mkdir(parents=True, exist_ok=True)

    if not (model_file.exists() and csv_file.exists()):
        print(f"✗ Missing model or CSV for {embedding_model_name}")
        continue

    model = BERTopic.load(str(model_file))
    df_topics = pd.read_csv(csv_file)
    topic_info_df = model.get_topic_info()

    summary_records = []

    for _, row in topic_info_df.iterrows():
        topic_id = int(row["Topic"])
        keywords = model.get_topic(topic_id)
        top_words = [kw for kw, _ in keywords[:10]]

        # Get documents for this topic
        topic_docs_df = df_topics[df_topics["topic_id"] == topic_id]
        topic_docs = topic_docs_df["text"].tolist()

        topic_data = {
            "topic_id": topic_id,
            "label": f"T{topic_id:03d}",
            "label_full": suggest_topic_name(top_words),
            "keywords": top_words,
            "n_docs": len(topic_docs),
            "documents": []
        }

        # Add documents to topic data
        for idx, doc_text in zip(topic_docs_df.index, topic_docs):
            topic_data["documents"].append({
                "document_id": int(idx),
                "score": None,
                "abstract": "\n".join(
                    [" ".join(doc_text.split()[j:j + 25]) for j in range(0, len(doc_text.split()), 25)]
                )
            })

        # Save individual topic JSON
        topic_file = topics_dir / f"topic_{topic_id:03d}.json"
        with open(topic_file, "w", encoding="utf-8") as f:
            json.dump(topic_data, f, ensure_ascii=False, indent=2)

        summary_records.append({
            "topic_id": topic_id,
            "label": f"T{topic_id:03d}",
            "label_full": suggest_topic_name(top_words),
            "n_docs": len(topic_docs),
            "top_words": ", ".join(top_words)
        })

        print(f"  T{topic_id:03d}: {top_words[:3]}")

    # Save summary files
    df_summary = pd.DataFrame(summary_records).sort_values("topic_id")
    csv_path = model_dir / "2_topic_summary.csv"
    latex_path = model_dir / "2_topic_summary.tex"
    
    df_summary.to_csv(csv_path, index=False)
    df_summary.to_latex(buf=latex_path, index=False, escape=False)

    print(f"✓ {len(summary_records)} topics exported")
    print(f"  → Topics JSON: {topics_dir}")
    print(f"  → Summary CSV: {csv_path}")


Exporting topics: all-MiniLM-L6-v2
  T-01: ['news', 'social', 'fake']
  T000: ['news', 'bias', 'participants']
  T001: ['stance', 'rumor', 'conversation']
  T002: ['medical', 'health', 'misinformation']
  T003: ['health', 'quality', 'videos']
  T004: ['reviews', 'review', 'online']
  T005: ['crowd', 'workers', 'crowdsourcing']
  T006: ['image', 'images', 'forgery']
  T007: ['credibility', 'articles', 'news']
  T008: ['fact', 'checking', 'knowledge']
  T009: ['evidence', 'claim', 'verification']
  T010: ['bert', 'news', 'fake']
  T011: ['multimodal', 'crossmodal', 'news']
  T012: ['sensing', 'discovery', 'observations']
  T013: ['blockchain', 'smart', 'reputation']
  T014: ['rumor', 'rumors', 'tweet']
  T015: ['face', 'deepfake', 'facial']
  T016: ['claim', 'task', 'factchecking']
  T017: ['news', 'fake', 'factbased']
  T018: ['news', 'entity', 'fake']
  T019: ['news', 'fake', 'dnfn']
  T020: ['image', 'compression', 'operations']
  T021: ['faces', 'videos', 'images']
  T022: ['news', 

In [5]:
# ============================================================================
# FILTER TOPICS & SAVE FILTERED DOCUMENTS
# ============================================================================

for embedding_model_name in embedding_model_names:
    print(f"\n{'='*60}")
    print(f"Filtering topics: {embedding_model_name}")
    print('='*60)

    normalized_model_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / normalized_model_name
    topics_dir = model_dir / "topics"

    topic_name_path = model_dir / "topic_names.json"
    if topic_name_path.exists():
        with open(topic_name_path, encoding="utf-8") as f:
            topic_names = {int(k): v for k, v in json.load(f).items()}
    else:
        topic_names = {}

    filtered_docs = []
    excluded_topics = {}

    # Process each topic JSON file
    for file_path in topics_dir.glob("*.json"):
        with file_path.open("r", encoding="utf-8") as f:
            topic_data = json.load(f)

        topic_id = topic_data.get("topic_id")
        top_words = topic_data.get("keywords", [])
        topic_label = topic_names.get(topic_id) or suggest_topic_name(top_words)

        documents = topic_data.get("documents", [])
        if topic_label:
            for doc in documents:
                filtered_docs.append({
                    "embedding_model": embedding_model_name,
                    "topic_id": topic_id,
                    "topic_label": topic_label,
                    "document_id": doc["document_id"],
                    "score": doc.get("score")
                })
        else:
            excluded_topics[topic_id] = top_words

    df_filtered = pd.DataFrame(filtered_docs)
    df_filtered.sort_values(by=["topic_id", "score"], ascending=[True, False], inplace=True, na_position='last')

    # Save filtered results
    df_filtered.to_json(model_dir / "3_documents_filtered_topic.json", orient="records", force_ascii=False, indent=2)
    df_filtered.to_csv(model_dir / "3_documents_filtered_topic.csv", index=False)

    with (model_dir / "4_excluded_topics.json").open("w", encoding="utf-8") as f:
        json.dump(dict(sorted(excluded_topics.items())), f, indent=2, ensure_ascii=False)

    print(f"✓ Included topics: {df_filtered['topic_id'].nunique()}")
    print(f"✓ Excluded topics: {len(excluded_topics)}")


Filtering topics: all-MiniLM-L6-v2
✓ Included topics: 58
✓ Excluded topics: 0

Filtering topics: all-mpnet-base-v2
✓ Included topics: 57
✓ Excluded topics: 0

Filtering topics: distiluse-base-multilingual-cased-v1
✓ Included topics: 58
✓ Excluded topics: 0

Filtering topics: paraphrase-MiniLM-L6-v2
✓ Included topics: 52
✓ Excluded topics: 0


In [2]:
# ============================================================================
# GENERATE UMAP VISUALIZATIONS & COMPUTE TOPIC COHESION
# ============================================================================

cohesion_percentile = 75
all_model_densities_bertopic = {}
model_percentile_thresholds_bertopic = {}
shared_umap_limits = {
    "x_min": float("inf"), "x_max": float("-inf"),
    "y_min": float("inf"), "y_max": float("-inf")
}
umap_results = {}

# First pass: Compute embeddings, cohesion, and store results
for embedding_model_name in embedding_model_names:
    print(f"\n{'='*60}")
    print(f"UMAP visualization: {embedding_model_name}")
    print('='*60)

    normalized_model_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / normalized_model_name
    figures_dir = model_dir / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    filtered_file = model_dir / "3_documents_filtered_topic.csv"
    model_file = model_dir / "bertopic_model.pkl"
    embeddings_file = model_dir / "documents_embeddings.npy"

    if not (filtered_file.exists() and model_file.exists() and embeddings_file.exists()):
        print(f"✗ Skipping — missing required files")
        continue

    # Load data
    df_filtered = pd.read_csv(filtered_file)
    df_filtered["document_id"] = df_filtered["document_id"].astype(int)
    df_filtered["topic_id"] = df_filtered["topic_id"].astype(int)

    topic_model = BERTopic.load(str(model_file))
    all_embeddings = np.load(embeddings_file)

    try:
        filtered_embeddings = np.array([all_embeddings[i] for i in df_filtered["document_id"]])
    except IndexError as e:
        print(f"✗ Index error: {e}")
        continue

    # Reduce to 2D for visualization
    reducer = UMAP(n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(filtered_embeddings)

    # Update global UMAP bounds
    shared_umap_limits["x_min"] = min(shared_umap_limits["x_min"], embedding_2d[:, 0].min())
    shared_umap_limits["x_max"] = max(shared_umap_limits["x_max"], embedding_2d[:, 0].max())
    shared_umap_limits["y_min"] = min(shared_umap_limits["y_min"], embedding_2d[:, 1].min())
    shared_umap_limits["y_max"] = max(shared_umap_limits["y_max"], embedding_2d[:, 1].max())

    # Compute topic cohesion (mean intra-topic distance)
    topic_densities = {}
    unique_topics = sorted(df_filtered["topic_id"].unique())
    all_scores = []

    for topic_id in unique_topics:
        indices = df_filtered[df_filtered["topic_id"] == topic_id].index
        coords = embedding_2d[indices]
        if len(coords) > 1:
            dist_matrix = pairwise_distances(coords)
            mean_dist = dist_matrix[np.triu_indices_from(dist_matrix, k=1)].mean()
        else:
            mean_dist = 0
        topic_densities[topic_id] = float(mean_dist)
        all_scores.append(mean_dist)

    all_model_densities_bertopic[embedding_model_name] = all_scores
    cohesion_threshold = np.percentile(all_scores, cohesion_percentile)
    model_percentile_thresholds_bertopic[embedding_model_name] = cohesion_threshold

    # Prepare data for plotting
    cmap = colormaps["tab20"]
    topic_to_color = {tid: cmap(i % 20) for i, tid in enumerate(unique_topics)}

    df_filtered["x"] = embedding_2d[:, 0]
    df_filtered["y"] = embedding_2d[:, 1]
    df_filtered["density"] = df_filtered["topic_id"].map(topic_densities)
    df_filtered["color"] = df_filtered["topic_id"].map(topic_to_color)
    df_filtered["label"] = df_filtered.apply(lambda row: f"T{row['topic_id']}\n{row['density']:.2f}", axis=1)
    df_filtered["cohesive"] = df_filtered["density"] < cohesion_threshold

    # Save processed data
    df_filtered.to_csv(model_dir / "4_umap_coords.csv", index=False)

    df_cohesion = pd.DataFrame({
        "topic_id": list(topic_densities.keys()),
        "density": list(topic_densities.values())
    })
    df_cohesion.to_csv(model_dir / "4_topic_cohesion.csv", index=False)

    # Save cohesion histogram
    plt.figure(figsize=(6, 4))
    plt.hist(all_scores, bins=30, edgecolor='black', alpha=0.7)
    plt.axvline(cohesion_threshold, color="red", linestyle="--", linewidth=2, label=f"Threshold: {cohesion_threshold:.2f}")
    plt.title(f"Topic Cohesion Distribution")
    plt.xlabel("Mean Intra-topic Distance")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(figures_dir / "topic_cohesion_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Save embeddings and bounds
    np.save(model_dir / "4_umap_embedding.npy", embedding_2d)
    with open(model_dir / "4_umap_bounds.json", "w") as f:
        json.dump({k: float(v) for k, v in shared_umap_limits.items()}, f, indent=2)

    # Store for plotting phase
    umap_results[embedding_model_name] = (df_filtered, embedding_2d, cohesion_threshold, topic_to_color)
    print(f"✓ Processed {len(unique_topics)} topics")

# Second pass: Generate visualizations
for embedding_model_name, (df_filtered, embedding_2d, cohesion_threshold, topic_to_color) in umap_results.items():
    print(f"\n  Plotting {embedding_model_name}...")

    normalized_model_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / normalized_model_name
    figures_dir = model_dir / "figures"

    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_densities = df_filtered.groupby("topic_id")["density"].first().to_dict()

    # Plot 1: All Topics (Global Axes)
    plt.figure(figsize=(6, 4))
    for tid in unique_topics:
        coords = df_filtered[df_filtered["topic_id"] == tid][["x", "y"]].values
        x, y = coords[:, 0], coords[:, 1]
        label = df_filtered[df_filtered["topic_id"] == tid]["topic_label"].iloc[0] if "topic_label" in df_filtered.columns else f"T{tid}"
        plt.scatter(x, y, color=topic_to_color[tid], label=f"T{tid}: {label}", s=20, alpha=0.7)
    plt.xlim(shared_umap_limits["x_min"], shared_umap_limits["x_max"])
    plt.ylim(shared_umap_limits["y_min"], shared_umap_limits["y_max"])
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("All Topics (Global Axes)")
    plt.tight_layout()
    plt.savefig(figures_dir / "filtered_topics_umap_all_fixed_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Plot 2: All Topics (Local Axes)
    plt.figure(figsize=(6, 4))
    for tid in unique_topics:
        coords = df_filtered[df_filtered["topic_id"] == tid][["x", "y"]].values
        x, y = coords[:, 0], coords[:, 1]
        plt.scatter(x, y, color=topic_to_color[tid], s=20, alpha=0.7)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("All Topics (Local Axes)")
    plt.tight_layout()
    plt.savefig(figures_dir / "filtered_topics_umap_all_local_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Plot 3: Cohesive Topics Only (Global Axes)
    retained_df = df_filtered[df_filtered["cohesive"]].copy()
    plt.figure(figsize=(6, 4))
    for tid in retained_df["topic_id"].unique():
        group = retained_df[retained_df["topic_id"] == tid]
        x, y = group["x"].values, group["y"].values
        plt.scatter(x, y, color=group["color"].iloc[0], s=20, alpha=0.7)
    plt.xlim(shared_umap_limits["x_min"], shared_umap_limits["x_max"])
    plt.ylim(shared_umap_limits["y_min"], shared_umap_limits["y_max"])
    n_retained = retained_df["topic_id"].nunique()
    n_total = len(topic_densities)
    plt.title(f"Cohesive Topics: {n_retained}/{n_total} (p={cohesion_percentile})")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(figures_dir / "filtered_topics_umap_cohesive_fixed_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Plot 4: Cohesive Topics Only (Local Axes)
    plt.figure(figsize=(6, 4))
    for tid in retained_df["topic_id"].unique():
        group = retained_df[retained_df["topic_id"] == tid]
        x, y = group["x"].values, group["y"].values
        plt.scatter(x, y, color=group["color"].iloc[0], s=20, alpha=0.7)
    plt.title(f"Cohesive Topics: {n_retained}/{n_total} (p={cohesion_percentile})")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(figures_dir / "filtered_topics_umap_cohesive_local_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Plot 5: Topic Size CCDF
    topic_sizes = df_filtered["topic_id"].value_counts().sort_values(ascending=False).values
    sorted_sizes = np.sort(topic_sizes)
    ccdf = 1.0 - np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)

    df_ccdf = pd.DataFrame({
        "sorted_sizes": sorted_sizes,
        "ccdf": ccdf,
        "embedding": embedding_model_name,
        "model": "BERTopic"
    })
    df_ccdf.to_parquet(model_dir / f"CCDF_BERTopic_{embedding_model_name}.parquet")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.step(sorted_sizes, ccdf, where="post", linewidth=2)
    ax.set_xlabel("Topic Size (Documents)")
    ax.set_ylabel("P(Size ≥ x)")
    ax.set_title("Topic Size CCDF")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(figures_dir / "topic_size_ccdf.png", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()

    # Save stats
    retained_df.to_csv(model_dir / "4_umap_coords_cohesive.csv", index=False)
    with open(model_dir / "4_umap_stats.json", "w") as f:
        json.dump({
            "cohesion_threshold": float(cohesion_threshold),
            "n_retained_topics": int(n_retained),
            "n_total_topics": int(n_total),
            "x_min": float(shared_umap_limits["x_min"]),
            "x_max": float(shared_umap_limits["x_max"]),
            "y_min": float(shared_umap_limits["y_min"]),
            "y_max": float(shared_umap_limits["y_max"])
        }, f, indent=2)

    print(f"  ✓ {embedding_model_name} visualization complete")


UMAP visualization: all-MiniLM-L6-v2
✓ Processed 58 topics

UMAP visualization: all-mpnet-base-v2
✓ Processed 57 topics

UMAP visualization: distiluse-base-multilingual-cased-v1
✓ Processed 58 topics

UMAP visualization: paraphrase-MiniLM-L6-v2
✓ Processed 52 topics

  Plotting all-MiniLM-L6-v2...
  ✓ all-MiniLM-L6-v2 visualization complete

  Plotting all-mpnet-base-v2...
  ✓ all-mpnet-base-v2 visualization complete

  Plotting distiluse-base-multilingual-cased-v1...
  ✓ distiluse-base-multilingual-cased-v1 visualization complete

  Plotting paraphrase-MiniLM-L6-v2...
  ✓ paraphrase-MiniLM-L6-v2 visualization complete


In [3]:
# ============================================================================
# STATISTICAL TEST: KRUSKAL-WALLIS
# ============================================================================

valid_models = {
    name: [float(v) for v in vals if np.isfinite(v)]
    for name, vals in all_model_densities_bertopic.items()
    if sum(np.isfinite(vals)) > 1
}

if len(valid_models) < 2:
    print("⚠ Insufficient data for Kruskal-Wallis test")
else:
    groups = list(valid_models.values())
    h_stat, p_value = kruskal(*groups)

    print(f"\n{'='*60}")
    print("Kruskal-Wallis Test: Topic Cohesion Across Embedding Models")
    print('='*60)
    print(f"H-statistic: {h_stat:.4f}")
    print(f"p-value:     {p_value:.4e}")
    print(f"Models tested: {len(groups)}")
    
    if p_value < 0.01:
        print("→ Strong evidence of significant difference (p < 0.01)")
    elif p_value < 0.05:
        print("→ Moderate evidence of difference (p < 0.05)")
    else:
        print("→ No significant difference detected (p ≥ 0.05)")


Kruskal-Wallis Test: Topic Cohesion Across Embedding Models
H-statistic: 21.2192
p-value:     9.4799e-05
Models tested: 4
→ Strong evidence of significant difference (p < 0.01)


In [4]:
# ============================================================================
# COMPOSITE VISUALIZATION: UMAP PLOTS
# ============================================================================

def build_composite_umap_sheet(suffix_all, suffix_cohesive, output_name):
    """Build a composite figure with UMAP plots from all models."""
    plot_pairs = []
    
    for model_name in embedding_model_names:
        norm_name = model_name.replace("/", "_")
        figures_dir = results_dir / norm_name / "figures"
        img_all = figures_dir / f"filtered_topics_umap_all_{suffix_all}.png"
        img_cohesive = figures_dir / f"filtered_topics_umap_cohesive_{suffix_cohesive}.png"
        
        if img_all.exists() and img_cohesive.exists():
            plot_pairs.append((model_name, img_all, img_cohesive))

    if not plot_pairs:
        print("⚠ No UMAP plots found")
        return

    n_models = len(plot_pairs)
    fig_width = 3 * n_models
    fig_height = 6

    fig, axes = plt.subplots(2, n_models, figsize=(fig_width, fig_height))
    axes = np.atleast_2d(axes)

    for col_idx, (model_name, img_all, img_cohesive) in enumerate(plot_pairs):
        threshold = model_percentile_thresholds_bertopic.get(model_name, float('nan'))

        # Row 0: All topics
        img = Image.open(img_all)
        axes[0][col_idx].imshow(img)
        axes[0][col_idx].axis("off")
        axes[0][col_idx].set_title(f"{model_name}\n(All Topics)", fontsize=10, weight='bold')

        # Row 1: Cohesive topics
        img = Image.open(img_cohesive)
        axes[1][col_idx].imshow(img)
        axes[1][col_idx].axis("off")
        axes[1][col_idx].set_title(f"{model_name}\n(Cohesive Topics)", fontsize=10, weight='bold')

    plt.tight_layout()
    out_path = results_dir / output_name
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved: {output_name}")


# Generate composite sheets
build_composite_umap_sheet(
    suffix_all="fixed_axes",
    suffix_cohesive="fixed_axes",
    output_name="composite_umap_global_axes.png"
)

build_composite_umap_sheet(
    suffix_all="local_axes",
    suffix_cohesive="local_axes",
    output_name="composite_umap_local_axes.png"
)

✓ Saved: composite_umap_global_axes.png
✓ Saved: composite_umap_local_axes.png


In [5]:
# ============================================================================
# COMPOSITE VISUALIZATION: TOPIC SIZE CCDF
# ============================================================================

ccdf_images = []
for model_name in embedding_model_names:
    norm_name = model_name.replace("/", "_")
    fig_path = results_dir / norm_name / "figures" / "topic_size_ccdf.png"
    if fig_path.exists():
        ccdf_images.append((model_name, fig_path))

if ccdf_images:
    n_cols = len(ccdf_images)
    fig_width = 4 * n_cols
    fig_height = 3

    fig, axes = plt.subplots(1, n_cols, figsize=(fig_width, fig_height), constrained_layout=True)
    fig.patch.set_facecolor("white")
    axes = np.atleast_1d(axes)

    for i, (model_name, img_path) in enumerate(ccdf_images):
        img = Image.open(img_path)
        axes[i].imshow(img, interpolation="none")
        axes[i].axis("off")
        axes[i].set_title(f"{model_name}", fontsize=11, weight='bold', pad=8)

    out_path = results_dir / "composite_ccdf_topic_sizes.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"✓ Saved: {out_path}")
else:
    print("⚠ No CCDF plots found")

✓ Saved: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/gpt-oss-120b/topic-modeling/bertopic/results/composite_ccdf_topic_sizes.png


In [6]:
# ============================================================================
# TOPIC COHERENCE: GENSIM C_V METRIC
# ============================================================================

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

cohesive_cv_summary = []

for embedding_model_name in embedding_model_names:
    print(f"\n{'='*60}")
    print(f"Computing C_V coherence: {embedding_model_name}")
    print('='*60)

    norm_name = embedding_model_name.replace("/", "_")
    model_dir = results_dir / norm_name
    model_file = model_dir / "bertopic_model.pkl"
    cohesive_file = model_dir / "4_umap_coords_cohesive.csv"

    if not (model_file.exists() and cohesive_file.exists()):
        print(f"✗ Missing model or cohesive data")
        continue

    # Load model and cohesive documents
    topic_model = BERTopic.load(str(model_file))
    retained_df = pd.read_csv(cohesive_file)

    cohesive_topic_ids = set(retained_df["topic_id"].unique())
    cohesive_doc_ids = retained_df["document_id"].astype(int).unique().tolist()

    # Tokenize documents using same vectorizer
    analyser = topic_model.vectorizer_model.build_analyzer()
    tokenised_docs = [analyser(text_series[i]) for i in cohesive_doc_ids]

    if not tokenised_docs:
        print(f"✗ No documents to analyze")
        continue

    # Create dictionary and extract topic words
    dictionary = Dictionary(tokenised_docs)

    top_n = 10
    topic_words = [
        [w for w, _ in topic_model.get_topic(tid)[:top_n]]
        for tid in sorted(cohesive_topic_ids) if tid != -1
    ]

    # Compute C_V coherence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenised_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    per_topic_scores = cm.get_coherence_per_topic()
    cv_overall = float(np.mean(per_topic_scores)) if per_topic_scores else float("nan")

    cohesive_cv_summary.append({
        "embedding_model": embedding_model_name,
        "n_cohesive_topics": len(cohesive_topic_ids),
        "cv_overall": cv_overall
    })

    print(f"C_V Score: {cv_overall:.4f} | Topics: {len(cohesive_topic_ids)}")

# Save summary
df_cv = pd.DataFrame(cohesive_cv_summary)
out_path = results_dir / "coherence_summary.csv"
df_cv.to_csv(out_path, index=False)
print(f"\n{'='*60}")
print(f"✓ Coherence scores saved to: {out_path}")
print(f"{'='*60}")


Computing C_V coherence: all-MiniLM-L6-v2
C_V Score: 0.5097 | Topics: 43

Computing C_V coherence: all-mpnet-base-v2
C_V Score: 0.5301 | Topics: 42

Computing C_V coherence: distiluse-base-multilingual-cased-v1
C_V Score: 0.4527 | Topics: 43

Computing C_V coherence: paraphrase-MiniLM-L6-v2
C_V Score: 0.4955 | Topics: 39

✓ Coherence scores saved to: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/gpt-oss-120b/topic-modeling/bertopic/results/coherence_summary.csv
